# PSSN Week 2 — Semantic-state segmentation

Run cells in order in Google Colab. This notebook reads the completed-correct traces from the W2-001 run, loads only the Qwen3-4B tokenizer (not model weights), segments each trace deterministically, and writes a timestamped output folder next to the source run. Original trace files are never overwritten. If the repository clone does not yet contain the new segmenter, the import cell lets you upload the four `src/semantic_states/*.py` files.

In [ ]:
from pathlib import Path
import os, subprocess, sys

REPOSITORY_URL = "https://github.com/Rami2212/PSSN-LLM-Reasoning.git"
REPOSITORY_DIR = Path("/content/PSSN-LLM-Reasoning")
if not (REPOSITORY_DIR / "requirements.txt").exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(REPOSITORY_DIR)], check=True)
os.chdir(REPOSITORY_DIR)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
from google.colab import drive
drive.mount("/content/drive")
print("Repository:", Path.cwd())

In [ ]:
# If the current GitHub checkout lacks the W2-003 implementation, upload
# __init__.py, schema.py, segmenter.py, and tokenizer_utils.py from src/semantic_states.
try:
    from src.semantic_states import UnsegmentableTraceError, segment_reasoning_trace
    from src.semantic_states.tokenizer_utils import load_qwen_tokenizer
    print("Semantic-state segmenter is available in the checkout.")
except ImportError:
    from google.colab import files
    import re
    uploaded = files.upload()
    required = {"__init__.py", "schema.py", "segmenter.py", "tokenizer_utils.py"}
    normalized_uploads = {}
    for uploaded_name, content in uploaded.items():
        canonical_name = re.sub(r" \(\d+\)(?=\.py$)", "", Path(uploaded_name).name)
        if canonical_name in required:
            normalized_uploads[canonical_name] = content
    missing = required - set(normalized_uploads)
    if missing:
        raise ValueError(f"Upload all four semantic_states files; missing: {sorted(missing)}")
    package_dir = Path("src/semantic_states")
    package_dir.mkdir(parents=True, exist_ok=True)
    for filename in required:
        (package_dir / filename).write_bytes(normalized_uploads[filename])
    from src.semantic_states import UnsegmentableTraceError, segment_reasoning_trace
    from src.semantic_states.tokenizer_utils import load_qwen_tokenizer
    print("Uploaded the current semantic-state implementation.")

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import json, statistics, subprocess

MODEL_ID = "Qwen/Qwen3-4B"
ARTIFACT_ROOT = Path("/content/drive/MyDrive/PSSN2/artifacts/expanded_baseline")
RUN_DIRECTORY_NAME = None  # Set to a specific run folder name, or keep None for latest.
if RUN_DIRECTORY_NAME is None:
    candidate_files = sorted(ARTIFACT_ROOT.glob("*/segmentation_ready_traces.jsonl"))
    if not candidate_files:
        raise FileNotFoundError(f"No segmentation_ready_traces.jsonl found under {ARTIFACT_ROOT}")
    source_path = candidate_files[-1]
else:
    source_path = ARTIFACT_ROOT / RUN_DIRECTORY_NAME / "segmentation_ready_traces.jsonl"
if not source_path.is_file():
    raise FileNotFoundError(source_path)
source_run_dir = source_path.parent
source_records = [json.loads(line) for line in source_path.read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Source run: {source_run_dir.name} ({len(source_records)} traces)")

tokenizer = load_qwen_tokenizer(MODEL_ID)
tokenizer_revision = (getattr(tokenizer, "init_kwargs", None) or {}).get("_commit_hash")

segmented_records, review_records = [], []
for record in source_records:
    try:
        states = segment_reasoning_trace(record, tokenizer)
        segmented_records.extend(state.to_dict() for state in states)
    except UnsegmentableTraceError as exc:
        review_records.append({"problem_id": record.get("problem_id"), "run_id": record.get("run_id"), "error": str(exc)})

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
output_dir = source_run_dir / "semantic_state_segmentation" / stamp
output_dir.mkdir(parents=True, exist_ok=False)
states_path = output_dir / "semantic_states.jsonl"
review_path = output_dir / "segmentation_review.jsonl"
summary_path = output_dir / "segmentation_summary.json"
states_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in segmented_records), encoding="utf-8")
review_path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in review_records), encoding="utf-8")
counts = [row["token_count"] for row in segmented_records]
try:
    repository_commit = subprocess.run(["git", "rev-parse", "HEAD"], check=True, capture_output=True, text=True).stdout.strip()
except Exception:
    repository_commit = None
summary = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_file": str(source_path),
    "source_run_id": source_records[0].get("run_id") if source_records else None,
    "source_trace_count": len(source_records),
    "segmented_trace_count": len({row["trace_id"] for row in segmented_records}),
    "state_count": len(segmented_records),
    "review_trace_count": len(review_records),
    "tokenizer_id": MODEL_ID,
    "tokenizer_revision": tokenizer_revision,
    "transformers_version": version("transformers"),
    "repository_commit": repository_commit,
    "state_token_count_min": min(counts) if counts else None,
    "state_token_count_median": statistics.median(counts) if counts else None,
    "state_token_count_max": max(counts) if counts else None,
    "output_files": {"states": str(states_path), "review": str(review_path)},
}
summary_path.write_text(json.dumps(summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(summary, indent=2))

In [ ]:
# Inspect several generated states and any traces flagged for review.
print("Sample states:")
for row in segmented_records[:10]:
    print(f"{row['problem_id']} | {row['state_id']} | {row['token_count']} tokens | final={row['is_final_state']}")
    print(row['text'][:350])
if review_records:
    print("\nTraces requiring manual review:")
    for row in review_records:
        print(row)
else:
    print("\nAll source traces were segmented.")